# ML3. Часть 3. Data splits / Разбиения данных

В этом ноутбуке реализованы методы из третьей части задания:

1. случайное разбиение данных на `train` и `test` с параметром `test_size`;
2. случайное разбиение данных на `train`, `validation` и `test` с параметрами `validation_size` и `test_size`;
3. разбиение данных на `train` и `test` по дате `date_split`;
4. разбиение данных на `train`, `validation` и `test` по датам `validation_date` и `test_date`;
5. объяснение, что значит сделать процедуру разбиения детерминированной.

Для случайных разбиений используется `RANDOM_STATE = 42`, выбранный из разрешенного списка: `21, 38, 42, 1, 0`.

## 1. Импорт библиотек и загрузка данных

Используем библиотеки из списка проекта: `pandas`, `numpy`, `sklearn`. Для разбиений нам нужен `train_test_split` из `sklearn.model_selection`.

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 21
DATA_PATH = Path(r".\data\train_df_processed_0.75.csv")

df = pd.read_csv(DATA_PATH)
df["created"] = pd.to_datetime(df["created"])

print(f"Размер датасета: {df.shape}")
display(df.head())
display(df.dtypes)

Размер датасета: (36284, 24)


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
0,1.0,0,2725,0,2016-05-26 15:01:42,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1.0,3,5200,0,2016-06-16 05:42:00,1,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,1.0,2,4204,1,2016-04-20 02:00:53,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,1.0,1,3500,0,2016-04-22 05:35:23,1,0,0,0,1,...,0,0,1,0,1,0,0,1,0,0
4,1.0,1,2195,0,2016-05-20 01:22:06,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


bathrooms                   float64
bedrooms                      int64
price                         int64
interest                      int64
created              datetime64[us]
Elevator                      int64
HardwoodFloors                int64
CatsAllowed                   int64
DogsAllowed                   int64
Doorman                       int64
Dishwasher                    int64
NoFee                         int64
LaundryinBuilding             int64
FitnessCenter                 int64
Pre-War                       int64
LaundryinUnit                 int64
RoofDeck                      int64
OutdoorSpace                  int64
DiningRoom                    int64
HighSpeedInternet             int64
Balcony                       int64
SwimmingPool                  int64
NewConstruction               int64
Terrace                       int64
dtype: object

Посмотрим минимальную и максимальную дату в колонке `created`. Это поможет выбрать адекватные даты для `date_split`, `validation_date` и `test_date`.

In [26]:
print("Минимальная дата:", df["created"].min())
print("Максимальная дата:", df["created"].max())

display(
    df["created"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .rename("objects_count")
    .to_frame()
)

Минимальная дата: 2016-04-01 22:12:41
Максимальная дата: 2016-06-29 21:41:47


,objects_count
created,
2016-04,12097
2016-05,11581
2016-06,12606


## 2. Вспомогательные функции проверки параметров

Перед реализацией разбиений удобно добавить маленькие проверки:

* `test_size` и `validation_size` должны быть долями от 0 до 1;
* сумма `validation_size + test_size` должна быть меньше 1, иначе для `train` ничего не останется;
* для временного разбиения `validation_date` должна быть раньше `test_date`.

Такие проверки делают функции более надежными: если пользователь передаст некорректные параметры, ошибка будет понятной.

In [27]:
def validate_ratio(name: str, value: float) -> None:
    """Проверяет, что значение является долей строго между 0 и 1."""
    if not 0 < value < 1:
        raise ValueError(f"{name} должен быть долей строго между 0 и 1. Получено: {value}")


def validate_dataframe(data: pd.DataFrame) -> None:
    """Проверяет, что на вход передан непустой DataFrame."""
    if not isinstance(data, pd.DataFrame):
        raise TypeError("data должен быть pandas DataFrame")
    if data.empty:
        raise ValueError("data не должен быть пустым")

## 3. Случайное разбиение на 2 части: train/test

Задание:

> Split data into 2 parts randomly with parameter `test_size` (ratio from 0 to 1), return training and test samples.

Функция ниже случайно делит датасет на две части:

* `train` - обучающая часть;
* `test` - тестовая часть.

Параметр `test_size` задает долю объектов, которые попадут в `test`. Например, `test_size=0.25` означает, что 25% данных попадет в `test`, а 75% - в `train`.

Для детерминированности используется `random_state=42`.

In [28]:
def split_random_train_test(
    data: pd.DataFrame,
    test_size: float,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Случайно делит DataFrame на train и test."""
    validate_dataframe(data)
    validate_ratio("test_size", test_size)

    train, test = train_test_split(
        data,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )

    return train.copy(), test.copy()

In [29]:
train_random, test_random = split_random_train_test(df, test_size=0.25)

print("train_random:", train_random.shape)
print("test_random:", test_random.shape)
print("Доля test:", round(len(test_random) / len(df), 4))

train_random: (27213, 24)
test_random: (9071, 24)
Доля test: 0.25


## 4. Случайное разбиение на 3 части: train/validation/test

Задание:

> Randomly split data into 3 parts with parameters `validation_size` and `test_size`, return train, validation and test samples.

Здесь `validation_size` и `test_size` трактуются как доли от полного датасета.

Например:

* `validation_size=0.20`;
* `test_size=0.20`.

Тогда итоговое разбиение должно быть примерно таким:

* `train` - 60%;
* `validation` - 20%;
* `test` - 20%.

Технически удобно делать это в два шага:

1. сначала отделить `test` от полного датасета;
2. затем из оставшейся части отделить `validation`.

Так как на втором шаге мы делим уже не полный датасет, долю `validation` нужно пересчитать:

`adjusted_validation_size = validation_size / (1 - test_size)`

In [30]:
def split_random_train_valid_test(
    data: pd.DataFrame,
    validation_size: float,
    test_size: float,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Случайно делит DataFrame на train, validation и test."""
    validate_dataframe(data)
    validate_ratio("validation_size", validation_size)
    validate_ratio("test_size", test_size)

    if validation_size + test_size >= 1:
        raise ValueError("Сумма validation_size и test_size должна быть меньше 1")

    train_valid, test = train_test_split(
        data,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )

    adjusted_validation_size = validation_size / (1 - test_size)

    train, validation = train_test_split(
        train_valid,
        test_size=adjusted_validation_size,
        random_state=random_state,
        shuffle=True,
    )

    return train.copy(), validation.copy(), test.copy()

In [31]:
train_random_3, valid_random_3, test_random_3 = split_random_train_valid_test(
    df,
    validation_size=0.20,
    test_size=0.20,
)

print("train_random_3:", train_random_3.shape, round(len(train_random_3) / len(df), 4))
print("valid_random_3:", valid_random_3.shape, round(len(valid_random_3) / len(df), 4))
print("test_random_3:", test_random_3.shape, round(len(test_random_3) / len(df), 4))

train_random_3: (21770, 24) 0.6
valid_random_3: (7257, 24) 0.2
test_random_3: (7257, 24) 0.2


## 5. Разбиение на 2 части по дате: train/test

Задание:

> Split data into 2 parts with parameter `date_split`, return train and test samples split by `date_split` param.

Такой подход нужен, когда в данных есть временная зависимость. В этом случае случайное перемешивание может дать слишком оптимистичную оценку качества, потому что модель увидит будущие паттерны в train.

Логика разбиения:

* все объекты с датой раньше `date_split` попадают в `train`;
* все объекты с датой `date_split` или позже попадают в `test`.

Это похоже на реальный production-сценарий: модель обучается на прошлом и проверяется на будущем.

In [32]:
def split_by_date_train_test(
    data: pd.DataFrame,
    date_split: str | pd.Timestamp,
    date_field: str = "created",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит DataFrame на train и test по одной дате."""
    validate_dataframe(data)

    if date_field not in data.columns:
        raise KeyError(f"В DataFrame нет колонки {date_field!r}")

    prepared = data.copy()
    prepared[date_field] = pd.to_datetime(prepared[date_field])
    date_split = pd.to_datetime(date_split)

    train = prepared[prepared[date_field] < date_split].copy()
    test = prepared[prepared[date_field] >= date_split].copy()

    return train, test

In [33]:
train_date, test_date = split_by_date_train_test(
    df,
    date_split="2016-06-01",
    date_field="created",
)

print("train_date:", train_date.shape)
print("test_date:", test_date.shape)
print("Максимальная дата в train:", train_date["created"].max())
print("Минимальная дата в test:", test_date["created"].min())

train_date: (23678, 24)
test_date: (12606, 24)
Максимальная дата в train: 2016-05-31 23:10:48
Минимальная дата в test: 2016-06-01 01:10:37


## 6. Разбиение на 3 части по датам: train/validation/test

Задание:

> Split data into 3 parts with parameters `validation_date` and `test_date`, return train, validation and test samples split by input params.

Логика разбиения:

* `train`: объекты раньше `validation_date`;
* `validation`: объекты от `validation_date` включительно до `test_date` не включительно;
* `test`: объекты от `test_date` включительно и позже.

Это полезно для настройки модели на более позднем периоде, чем train, и финальной проверки на самом новом периоде.

In [34]:
def split_by_date_train_valid_test(
    data: pd.DataFrame,
    validation_date: str | pd.Timestamp,
    test_date: str | pd.Timestamp,
    date_field: str = "created",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Делит DataFrame на train, validation и test по двум датам."""
    validate_dataframe(data)

    if date_field not in data.columns:
        raise KeyError(f"В DataFrame нет колонки {date_field!r}")

    validation_date = pd.to_datetime(validation_date)
    test_date = pd.to_datetime(test_date)

    if validation_date >= test_date:
        raise ValueError("validation_date должна быть раньше test_date")

    prepared = data.copy()
    prepared[date_field] = pd.to_datetime(prepared[date_field])

    train = prepared[prepared[date_field] < validation_date].copy()
    validation = prepared[
        (prepared[date_field] >= validation_date)
        & (prepared[date_field] < test_date)
    ].copy()
    test = prepared[prepared[date_field] >= test_date].copy()

    return train, validation, test

In [35]:
train_date_3, valid_date_3, test_date_3 = split_by_date_train_valid_test(
    df,
    validation_date="2016-05-01",
    test_date="2016-06-01",
    date_field="created",
)

print("train_date_3:", train_date_3.shape)
print("valid_date_3:", valid_date_3.shape)
print("test_date_3:", test_date_3.shape)

print("train date range:", train_date_3["created"].min(), "-", train_date_3["created"].max())
print("valid date range:", valid_date_3["created"].min(), "-", valid_date_3["created"].max())
print("test date range:", test_date_3["created"].min(), "-", test_date_3["created"].max())

train_date_3: (12097, 24)
valid_date_3: (11581, 24)
test_date_3: (12606, 24)
train date range: 2016-04-01 22:12:41 - 2016-04-30 19:21:03
valid date range: 2016-05-01 23:09:04 - 2016-05-31 23:10:48
test date range: 2016-06-01 01:10:37 - 2016-06-29 21:41:47


## 7. Детерминированность процедуры разбиения

Задание:

> Make split procedure determenistic. What does it mean?

Детерминированная процедура разбиения означает, что при одинаковых входных данных и одинаковых параметрах мы каждый раз получаем один и тот же результат.

Для разбиений это важно, потому что качество модели зависит от того, какие объекты попали в `train`, `validation` и `test`. Если каждый запуск дает новое разбиение, метрики будут меняться, и сравнивать эксперименты станет сложнее.

### Как сделать случайное разбиение детерминированным

В случайных разбиениях нужно зафиксировать `random_state`. В этом ноутбуке используется `RANDOM_STATE = 42`.

При фиксированном `random_state` функция `train_test_split` будет воспроизводимо перемешивать данные: индексы, попавшие в `train` и `test`, будут одинаковыми при каждом запуске.

### Как сделать разбиение по дате детерминированным

Разбиение по дате уже детерминированное, если:

* входной DataFrame одинаковый;
* значения в `date_field` одинаково преобразуются через `pd.to_datetime`;
* используются те же `date_split`, `validation_date` и `test_date`;
* правила границ явно зафиксированы, например `train < date_split`, `test >= date_split`.

То есть для временного разбиения случайность не нужна: результат полностью определяется датами и правилами сравнения.

In [36]:
train_a, test_a = split_random_train_test(df, test_size=0.25, random_state=RANDOM_STATE)
train_b, test_b = split_random_train_test(df, test_size=0.25, random_state=RANDOM_STATE)

print("Индексы train совпадают:", train_a.index.equals(train_b.index))
print("Индексы test совпадают:", test_a.index.equals(test_b.index))

Индексы train совпадают: True
Индексы test совпадают: True


## 8. Итог

В этом ноутбуке реализованы все методы из третьей части задания:

* `split_random_train_test` - случайное разбиение на `train` и `test`;
* `split_random_train_valid_test` - случайное разбиение на `train`, `validation` и `test`;
* `split_by_date_train_test` - разбиение на `train` и `test` по `date_split`;
* `split_by_date_train_valid_test` - разбиение на `train`, `validation` и `test` по `validation_date` и `test_date`.

Для случайных процедур зафиксирован `random_state=42`, поэтому разбиения воспроизводимы. Для временных процедур детерминированность обеспечивается фиксированными датами и строгими правилами попадания объектов в каждую часть.